# EV Purchases — CPU LightGBM baseline
Kaggle CPU only, no GPU. Runtime: ~3-5 min.

In [ ]:
!pip install -q lightgbm
import os, gc
import numpy as np, pandas as pd
import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
print(lgb.__version__)


In [ ]:
SEED=42; N_SPLITS=5
TRAIN_P='/kaggle/input/playground-series-s6e9/train.csv'
TEST_P='/kaggle/input/playground-series-s6e9/test.csv'
SUB_P='/kaggle/input/playground-series-s6e9/sample_submission.csv'
OUT='/kaggle/working/submission.csv'
CAT_COLS=['Gender','City_Type','Current_Car_Type','Home_Charging_Possible','Subsidy_Available','Range_Anxiety_Level']
TARGET='Will_Buy_EV'
tr=pd.read_csv(TRAIN_P); te=pd.read_csv(TEST_P)
y=(tr[TARGET]=='Yes').astype(int).values
X=tr.drop(columns=['id',TARGET]); Xte=te.drop(columns=['id'])
test_ids=te['id'].values
for c in CAT_COLS:
    X[c]=X[c].astype('category'); Xte[c]=Xte[c].astype('category')
print(tr.shape, te.shape, y.mean())


In [ ]:
params={'objective':'binary','metric':'auc','boosting_type':'gbdt','device_type':'cpu','verbosity':-1,'seed':SEED,'learning_rate':0.05,'num_leaves':63,'min_child_samples':200,'feature_fraction':0.8,'bagging_fraction':0.8,'bagging_freq':1,'lambda_l2':2.0}
skf=StratifiedKFold(n_splits=N_SPLITS,shuffle=True,random_state=SEED)
oof=np.zeros(len(X)); test_pred=np.zeros(len(Xte))
for fold,(ti,vi) in enumerate(skf.split(X,y)):
    print(f'--- fold {fold+1}/{N_SPLITS} ---')
    dtr=lgb.Dataset(X.iloc[ti],label=y[ti],categorical_feature=CAT_COLS)
    dva=lgb.Dataset(X.iloc[vi],label=y[vi],categorical_feature=CAT_COLS)
    m=lgb.train(params,dtr,num_boost_round=2000,valid_sets=[dtr,dva],callbacks=[lgb.early_stopping(100,verbose=50),lgb.log_evaluation(100)])
    oof[vi]=m.predict(X.iloc[vi]); test_pred+=m.predict(Xte)/N_SPLITS
    print(f'fold {fold+1} AUC={roc_auc_score(y[vi],oof[vi]):.5f} iter={m.best_iteration}')
print(f'OOF AUC={roc_auc_score(y,oof):.5f}')


In [ ]:
sub=pd.read_csv(SUB_P)
sub[TARGET]=test_pred
sub.to_csv(OUT,index=False)
print(OUT, sub.shape, sub[TARGET].mean())
sub.head()
